# NB03b — the filtered-training probe

**Off-plan. Decision doc: `claude_plans/filtering_retrain_decision.md` step 3. Budget ~5 h, single-GPU 80 GB pod.**

## The question this answers

`se/audit_dataset_balance.py` found that 43.0% of the pool patches at a prompt-template token
(`"or"`, `"else"`, `"and"`, `"with"`, `"Respond"`, `"nothing"`, `"one"`, `"of"`, `"assistant"`),
and that none of those map to the paper's five Table 7 categories. `se/stratify_by_position.py`
then showed that restricting *scoring* to content positions turns NB03's pooled near-zero lift
into a clearly positive one. That fix was free and is already applied.

What it does not fix is the **training** mixture. Every model in the tree learned from data
where 43% of examples carried an activation from a position with nothing to say, which
plausibly teaches a prior that the injected channel is noise — suppressing the effect under
study. If so, the measured content-token lift is a floor on the true one.

This notebook tests that, without invalidating anything:

> Train on content+relation rows only. Hold the eval set **byte-identical**. Compare the
> content-token lift under filtered training against the same quantity under unfiltered
> training, at matched N.

- **Materially larger under filtered training** → training dilution was suppressing the effect;
  the full retrain (~36 h+, whole tree invalidated) is justified, and this probe has already
  bounded its size.
- **Not materially larger** → the eval-side fix is sufficient. Report the stratified result,
  state the composition difference as a deviation from the paper, and skip the retrain.

## Why the eval set is not filtered

Filtering the whole dataset would shift the last-`EVAL_SIZE` window and break the join to every
run already on disk. Instead the filter is applied to the **train region only** and the eval
region is concatenated back untouched, so filtered and unfiltered arms are scored on identical
items in identical order and pair directly.

## Why the floor is retrained rather than reused

Reaching N=8,192 filtered rows requires walking ~14.4k raw rows instead of 8.2k, and the audit's
held-out leakage rises with depth (+0.124 at N=8,192, +0.148 at N=16,384). So the filtered arm
sees a slightly more leakage-rich prompt mixture. That shift is common to the floor and the
`Cfull` arms, so it cancels in the lift — but only if the floor is trained on the same mixture.
A reused unfiltered floor would silently absorb the leakage difference into the lift.

## Run tree

Everything here writes under `task="patching_filtered"`, a sibling of `patching` in the runs
tree. `run_dir` has no slot for "filtered" (`se_config.py:199`) — the task component is the only
one free, and using it keeps `bootstrap_paired.collect` from ever confusing the two trees.


In [1]:
# --- dependencies -----------------------------------------------------------
# Same bootstrap as NB03; safe to re-run, a no-op on a warm pod.
import os
import sys

REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()

dependencies: already satisfied (torch 2.8.0+cu128, transformers 5.15.0, peft 0.20.0, trl 1.10.0, bitsandbytes 0.50.1)


{}

In [2]:
# --- environment ------------------------------------------------------------
# Three training runs on an 8B explainer with a full-rank input map: same footprint as one
# NB03 arm, so the same pod spec.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C

RUNPOD ENVIRONMENT
host    : RunPod pod 6uqtvsxboal2o6  |  python 3.12.3
gpu     : 1 x NVIDIA A100 80GB PCIe  |  79 GiB  |  bf16 yes
torch   : 2.8.0+cu128  (CUDA 12.8)
stack   : transformers 5.15.0 · peft 0.20.0 · trl 1.10.0 · datasets 5.0.1 · accelerate 1.14.0 · bitsandbytes 0.50.1
repo    : /workspace/self_explainer
volume  : /workspace  (own mount, 461308 GiB free)
hf cache: /workspace/.cache/huggingface/
outputs : /workspace/self_explainer
hf token: none
!! no HF token found. Public checkpoints still work; set HF_TOKEN in the pod template or write it to /workspace/.hf_token if a download 401s.


## 1. Build the filtered datasets (CPU)

The filter is `audit_dataset_balance.bucket(token_type) != "prompt template"` — imported rather
than restated, so this notebook cannot drift from the taxonomy the audit and the stratification
already use.

Two things worth knowing about that taxonomy before trusting the split:

- `"relation_suffix"` (2,672 rows) is kept. It is the paper's own **Relation** category.
- `"unknown"` (1,685 rows, the third most common type) is kept as a content token, which looks
  wrong until you read a prompt: `"Respond with one of trance or disco or ... or unknown and
  nothing else"`. It is an **answer option**, not a missing value, so it maps to the paper's
  Orig/Other/Changed Answer Option categories. Dropping it would be the error.

The keep-indices are computed **once, from the identity arm, and applied to both arms**. The two
arms are row-aligned by construction (NB02) and this cell asserts it, but selecting by a shared
index list rather than filtering each arm independently makes the alignment structural instead
of contingent.

In [3]:
import json
import time

import numpy as np
import pandas as pd
from datasets import concatenate_datasets

import se_common as S
from audit_dataset_balance import TEMPLATE_TOKENS, bucket

PROBE_TASK = "patching_filtered"

# A three-point ladder, not a single point. The dilution hypothesis makes a SHAPE prediction --
# removing 43% dead examples should matter most where examples are scarcest -- so a DiD that
# grows as N falls is evidence and a flat one is not. One point at 8,192 cannot tell those
# apart. N=8,192 is also the ceiling: 10,817 content+relation rows sit outside the eval split,
# and 16,384 would need a pool prefix raise, which moves the eval window and invalidates the
# tree (n100k_scale_plan.md Appendix A).
PROBE_N = [512, 2048, 8192]

# One seed per point. The comparators all carry three-seed bands from NB03, and the reading is
# "is the filtered contrast outside that band", which a point estimate answers. The ladder is
# also the cheaper spend: 512+2048+8192 = 10,752 training examples per arm against 24,576 for
# three seeds at 8,192 alone -- three N values at one seed cost LESS than one N at three seeds
# and answer a question a single point cannot.
PROBE_SEEDS = [C.SEED]

tokenizer = S.load_tokenizer()

ds = {rot: S.load_ready_dataset(rot) for rot in ("identity", "Q")}
n_total = len(ds["identity"])
split = n_total - C.EVAL_SIZE
assert len(ds["Q"]) == n_total, "arms disagree on length — rebuild in NB02"

token_type = ds["identity"]["token_type"]
assert ds["Q"]["token_type"] == token_type, (
    "arms disagree on token_type row-for-row; a shared keep-index list would silently "
    "select different examples in each arm")

keep_train = [i for i in range(split) if bucket(token_type[i]) != "prompt template"]
need = max(PROBE_N)
assert len(keep_train) >= need, (
    f"only {len(keep_train):,} content+relation rows outside the eval split; "
    f"N={need:,} is unreachable without raising ACT_DATASET_PREFIX")

filtered = {rot: concatenate_datasets([d.select(keep_train), d.select(range(split, n_total))])
            for rot, d in ds.items()}

# The whole probe rests on this: the eval region must be the same items, in the same order, as
# every run already on disk. Assert it rather than trust the arithmetic.
for rot, d in filtered.items():
    assert len(d) == len(keep_train) + C.EVAL_SIZE
    tail_f = d.select(range(len(d) - C.EVAL_SIZE, len(d)))
    tail_u = ds[rot].select(range(split, n_total))
    assert tail_f["messages"] == tail_u["messages"], f"{rot}: eval window moved"
    assert tail_f["token_type"] == tail_u["token_type"], f"{rot}: eval window moved"

# v = 0 on the filtered mixture. Prompts, labels, ordering and budget identical to the filtered
# Cfull arms; only the activation's information is removed.
zeroed = filtered["identity"].map(lambda ex: {
    "patch_position": {**ex["patch_position"],
                       "intervention_vector": [0.0] * len(
                           ex["patch_position"]["intervention_vector"])},
})

probe_ds = {"zerovec": zeroed, "identity": filtered["identity"], "Q": filtered["Q"]}

# --- what the filter actually changed ---------------------------------------
def composition(idx):
    b = [bucket(token_type[i]) for i in idx]
    return {k: b.count(k) for k in ("content token", "relation", "prompt template")}

print(f"pool {n_total:,} rows   eval (last {C.EVAL_SIZE:,}) held fixed   "
      f"train region {split:,} rows")
print(f"content+relation outside eval : {len(keep_train):,} "
      f"({len(keep_train)/split:.1%} of the train region)")
print()
for n in PROBE_N:
    depth = keep_train[n - 1] + 1
    print(f"N={n:,}")
    print(f"  filtered   : walks {depth:,} raw rows   {composition(keep_train[:n])}")
    print(f"  unfiltered : walks {n:,} raw rows   {composition(range(n))}")
    print(f"  -> the filtered arm reaches {depth/n:.2f}x deeper into the pool; the audit's "
          f"held-out leakage rises with depth,")
    print(f"     which is why the floor is retrained on this same mixture rather than reused.")

pool 20,000 rows   eval (last 1,024) held fixed   train region 18,976 rows
content+relation outside eval : 10,817 (57.0% of the train region)

N=512
  filtered   : walks 885 raw rows   {'content token': 397, 'relation': 115, 'prompt template': 0}
  unfiltered : walks 512 raw rows   {'content token': 229, 'relation': 62, 'prompt template': 221}
  -> the filtered arm reaches 1.73x deeper into the pool; the audit's held-out leakage rises with depth,
     which is why the floor is retrained on this same mixture rather than reused.
N=2,048
  filtered   : walks 3,566 raw rows   {'content token': 1557, 'relation': 491, 'prompt template': 0}
  unfiltered : walks 2,048 raw rows   {'content token': 919, 'relation': 265, 'prompt template': 864}
  -> the filtered arm reaches 1.74x deeper into the pool; the audit's held-out leakage rises with depth,
     which is why the floor is retrained on this same mixture rather than reused.
N=8,192
  filtered   : walks 14,366 raw rows   {'content token': 6218

## 2. The three filtered runs

`floor`, `Cfull · R-id`, `Cfull · R-Q` — all trained on the filtered mixture, all writing under
`task="patching_filtered"`. `resume=True` is the default, so a preempted pod picks up where it
left off and re-running this cell is free.

In [4]:
# (label, rotation, capacity, init, dataset key)
PROBE_ARMS = [
    ("floor · filtered (v=0)", "zerovec",  "Cfull", "identity", "zerovec"),
    ("Cfull · R-id · filtered", "identity", "Cfull", "identity", "identity"),
    ("Cfull · R-Q  · filtered", "Q",        "Cfull", "identity", "Q"),
]

probe_results = []
for n in PROBE_N:
    for seed in PROBE_SEEDS:
        for label, rot, cap, init, dskey in PROBE_ARMS:
            t0 = time.time()
            print(f"\n=== {label} · n={n} · seed={seed} " + "=" * 24)
            kw = dict(tokenizer=tokenizer, dataset=probe_ds[dskey], seed=seed,
                      task=PROBE_TASK)
            S.run_training(n, rot, cap, init, **kw)
            scores = S.eval_run(n, rot, cap, init, **kw)
            scores["label"] = label
            probe_results.append(scores)
            print(f"  exact_match {scores['exact_match']:.3f} | "
                  f"has_changed_f1 {scores['has_changed_f1']:.3f} | "
                  f"content_match {scores['content_match']:.3f} "
                  f"[{(time.time()-t0)/60:.1f} min]")

probe_df = pd.DataFrame(probe_results)
probe_df.to_csv(f"{C.REPORTS_DIR}/filtered_probe_runs.csv", index=False)
print()
print(probe_df[["label", "n_train", "seed", "exact_match", "has_changed_f1", "content_match"]]
      .round(3).to_string(index=False))
print("\nThese are POOLED over eval positions and are not the reading. §3 restricts to content.")


=== floor · filtered (v=0) · n=512 · seed=67 ========================


Map:   0%|          | 0/11841 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

  input map: 67,108,864 trainable (expected 67,108,864), 67,108,864 frozen (PEFT's modules_to_save keeps the original alongside its trainable copy), LoRA-wrapped: no


Epoch,Training Loss,Validation Loss
1,0.781408,0.151733
2,0.502931,0.138694
3,0.314097,0.133828


...

Epoch,Training Loss,Validation Loss
1,0.453158,0.096548
2,0.257912,0.084166
3,0.126365,0.102899


Training Loss,Validation Loss,Epoch
0.126365,0.102899,3


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

  exact_match 0.627 | has_changed_f1 0.769 | content_match 0.738 [54.2 min]

                  label  n_train  seed  exact_match  has_changed_f1  content_match
 floor · filtered (v=0)      512    67        0.232           0.332          0.589
Cfull · R-id · filtered      512    67        0.245           0.356          0.591
Cfull · R-Q  · filtered      512    67        0.329           0.557          0.610
 floor · filtered (v=0)     2048    67        0.366           0.496          0.613
Cfull · R-id · filtered     2048    67        0.500           0.708          0.656
Cfull · R-Q  · filtered     2048    67        0.404           0.581          0.616
 floor · filtered (v=0)     8192    67        0.605           0.754          0.710
Cfull · R-id · filtered     8192    67        0.624           0.775          0.723
Cfull · R-Q  · filtered     8192    67        0.627           0.769          0.738

These are POOLED over eval positions and are not the reading. §3 restricts to content.


## 3. The reading: content-token lift, filtered vs unfiltered

A difference-in-differences over four quantities, all scored on the **same eval items** at
content positions only:

```
lift_unfiltered = score(patching,          Cfull R-Q) - score(patching,          floor)
lift_filtered   = score(patching_filtered, Cfull R-Q) - score(patching_filtered, floor)
DiD             = lift_filtered - lift_unfiltered
```

`DiD > 0` means training dilution was suppressing the effect.

The bootstrap draws **one item-index set per replicate and applies it to all four arms**, which
is what makes the interval paired — the eval-sampling variance that dominates an unpaired
interval cancels. Seeds are averaged within each arm before differencing; the unfiltered arms
have three, the filtered one, so seeds are not resampled here and the interval is item-only.
Read it against NB03's per-seed spread (~0.027 on `exact_match`) rather than as the whole
uncertainty.

In [5]:
from bootstrap_paired import METRICS, encode, metrics_on
from stratify_by_position import eval_position_labels


def load_task(task, n_train, rotation, capacity, init, seed):
    """bootstrap_paired.load, with the task component parameterized (it hardcodes 'patching')."""
    d = C.run_dir(task, rotation, capacity, init, n_train,
                  explainer=C.EXPLAINER_MODEL_ID, seed=seed, root=C.RUNS_DIR)
    p = f"{d}/eval_records.json"
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f)


def encs_for(task, n, arm, seeds):
    """Per-seed encodings for one arm, skipping seeds that have not run."""
    out = []
    for s in seeds:
        recs = load_task(task, n, *arm, s)
        if recs is not None:
            out.append((s, encode(recs)))
    return out


ARM_COORDS = {
    "floor": ("zerovec", "Cfull", "identity"),
    "R-id": ("identity", "Cfull", "identity"),
    "R-Q": ("Q", "Cfull", "identity"),
}

# (label, hi, lo) -- the contrast reported is hi - lo.
#
# Both are needed, and `delta` is the one that decides the retrain. `lift` is the activation's
# contribution and is what dilution would suppress; `delta` is the R-id/R-Q contrast the
# project's mechanism claim actually rests on. Filtered training could raise both lifts
# equally and leave every conclusion untouched -- that is a null for the retrain, however
# large the lift moved.
CONTRASTS = [
    ("lift  (R-Q  - floor)", "R-Q", "floor"),
    ("lift  (R-id - floor)", "R-id", "floor"),
    ("delta (R-Q  - R-id)", "R-Q", "R-id"),
]

labels, _ = eval_position_labels()
CONTENT = np.array([i for i, b in enumerate(labels) if b != "prompt template"], dtype=np.int64)
ALL = np.arange(len(labels), dtype=np.int64)
print(f"eval items: {len(ALL):,} total, {len(CONTENT):,} at content+relation positions "
      f"({len(CONTENT)/len(ALL):.1%})")


def seed_mean(encs, idx, metric):
    return float(np.mean([metrics_on(e, idx)[metric] for _, e in encs]))


def did_bootstrap(encs, hi, lo, idx, n_boot=10000, rng_seed=20260821):
    """95% CI on (filtered contrast - unfiltered contrast), one shared item draw per replicate.

    Every arm is resampled on the same indices, so the eval-set variance that dominates an
    unpaired interval cancels in the difference. All three metrics come out of one pass.
    """
    def contrasts(r):
        m = {k: [metrics_on(e, r) for _, e in v] for k, v in encs.items()}
        out = {}
        for metric in METRICS:
            mean = lambda k: float(np.mean([d[metric] for d in m[k]]))   # noqa: E731
            f = mean(("filtered", hi)) - mean(("filtered", lo))
            u = mean(("unfiltered", hi)) - mean(("unfiltered", lo))
            out[metric] = (f, u, f - u)
        return out

    rng = np.random.default_rng(rng_seed)
    point = contrasts(idx)
    draws = {metric: np.empty(n_boot) for metric in METRICS}
    for b in range(n_boot):
        rep = contrasts(idx[rng.integers(0, len(idx), len(idx))])
        for metric in METRICS:
            draws[metric][b] = rep[metric][2]
    return {metric: (*point[metric], *np.percentile(draws[metric], [2.5, 97.5]))
            for metric in METRICS}


report = {"task": PROBE_TASK, "n_content_items": int(len(CONTENT)), "by_n": {}}
for n in PROBE_N:
    unf_seeds = C.seeds_for(n)
    encs = {}
    for arm, coords in ARM_COORDS.items():
        encs[("filtered", arm)] = encs_for(PROBE_TASK, n, coords, PROBE_SEEDS)
        encs[("unfiltered", arm)] = encs_for("patching", n, coords, unf_seeds)
    missing = [k for k, v in encs.items() if not v]
    if missing:
        print(f"\nN={n:,}: SKIPPED -- no eval_records for {missing}. "
              f"The unfiltered arms come from NB03; run it first.")
        continue

    # The pairing is only valid if every arm scored the same items in the same order. This is
    # the one assumption the whole probe rests on, so it is a hard failure, not a warning.
    ref = encs[("unfiltered", "floor")][0][1]["target"]
    for k, v in encs.items():
        for s, e in v:
            assert e["target"] == ref, (
                f"{k} seed {s} scored different eval items -- the filtered dataset's eval "
                "window moved; re-run section 1 and check its asserts")

    print(f"\nN={n:,}   filtered seeds {[s for s, _ in encs[('filtered', 'R-Q')]]}   "
          f"unfiltered seeds {[s for s, _ in encs[('unfiltered', 'R-Q')]]}")

    # --- absolute scores first. If the DiD moves because the FLOOR moved, that is the pool-
    # depth/leakage effect, not dilution, and it is not a reason to retrain anything.
    print("=" * 92)
    print("ABSOLUTE exact_match at content positions")
    print(f"  {'arm':<10}{'unfiltered':>12}{'filtered':>11}{'shift':>9}")
    for arm in ARM_COORDS:
        u = seed_mean(encs[("unfiltered", arm)], CONTENT, "exact_match")
        f = seed_mean(encs[("filtered", arm)], CONTENT, "exact_match")
        print(f"  {arm:<10}{u:>12.3f}{f:>11.3f}{f - u:>+9.3f}")

    print("\nCONTRASTS -- filtered vs unfiltered, difference-in-differences")
    print("=" * 92)
    print(f"{'contrast':<22}{'metric':<16}{'pos':<10}{'unfilt':>9}{'filt':>9}"
          f"{'DiD':>9}   95% CI")
    rows = {}
    for label, hi, lo in CONTRASTS:
        for name, idx in (("content", CONTENT), ("all", ALL)):
            res = did_bootstrap(encs, hi, lo, idx)
            for metric in METRICS:
                f, u, did, clo, chi = res[metric]
                flag = "" if clo <= 0 <= chi else "   *"
                print(f"{label:<22}{metric:<16}{name:<10}{u:>+9.3f}{f:>+9.3f}"
                      f"{did:>+9.3f}   [{clo:+.3f}, {chi:+.3f}]{flag}")
                rows[f"{label}/{metric}/{name}"] = {
                    "unfiltered": u, "filtered": f, "did": did, "ci": [float(clo), float(chi)]}
        print()
    report["by_n"][str(n)] = rows
    print("  * = interval excludes zero (item resampling only; seeds are not resampled,")
    print("      so read it against NB03's per-seed spread of ~0.027 on exact_match)")

out = f"{C.REPORTS_DIR}/filtered_probe.json"
with open(out, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nwritten to {out}")

eval items: 1,024 total, 577 at content+relation positions (56.3%)

N=512   filtered seeds [67]   unfiltered seeds [67, 68, 69]
ABSOLUTE exact_match at content positions
  arm         unfiltered   filtered    shift
  floor            0.303      0.328   +0.025
  R-id             0.341      0.326   -0.016
  R-Q              0.388      0.367   -0.021

CONTRASTS -- filtered vs unfiltered, difference-in-differences
contrast              metric          pos          unfilt     filt      DiD   95% CI
lift  (R-Q  - floor)  exact_match     content      +0.085   +0.040   -0.046   [-0.090, -0.001]   *
lift  (R-Q  - floor)  has_changed_f1  content      +0.136   +0.201   +0.065   [+0.017, +0.111]   *
lift  (R-Q  - floor)  content_match   content      +0.036   +0.012   -0.023   [-0.063, +0.016]
lift  (R-Q  - floor)  exact_match     all          +0.056   +0.097   +0.040   [+0.008, +0.073]   *
lift  (R-Q  - floor)  has_changed_f1  all          +0.120   +0.225   +0.104   [+0.069, +0.138]   *
lift  (R-Q

## 4. Deciding

Two rows matter, both at `exact_match` / `content`:

- **`lift (R-Q - floor)`** — the activation's contribution. This is what training dilution would
  have suppressed.
- **`delta (R-Q - R-id)`** — the R-id/R-Q contrast. **This is the one that decides the retrain.**
  The project's mechanism claim is about this contrast, not about absolute lift. Filtered
  training could raise both lifts substantially and leave every conclusion in the writeup
  intact — a large `lift` DiD with a flat `delta` DiD is a *null* for the retrain question,
  however dramatic it looks.

The yardstick is NB03's per-seed spread: sd ~0.027 on `exact_match`, so a pooled seed band is
about **±0.033**. A DiD inside that is not a finding.

### Check the floor first

Read the ABSOLUTE table before any DiD. The filtered arm walks ~1.75x deeper into the pool at
N=8,192, and the audit's held-out leakage rises with depth (+0.124 at 8,192, +0.148 at 16,384).
If the filtered **floor** moved and both `Cfull` arms sat still, the lift grew for a reason that
has nothing to do with dilution, and no amount of retraining will reproduce it as a real effect.

### The rule

| Result at content positions | Reading | Action |
|---|---|---|
| `delta` DiD inside ±0.033 at every N | The contrast is unchanged by filtering | **No resweep.** Report the stratified result; record the composition difference as a stated deviation from the paper |
| `lift` DiD large, `delta` DiD inside the band | Dilution suppressed the magnitude but not the mechanism | **No resweep.** The absolute numbers are understated; say so in the writeup and cite this probe's size |
| `delta` DiD outside the band at a single N, no trend across the ladder | One point, one seed | **No resweep yet.** Add `C.seeds_for(n)` at that N before believing it |
| `delta` DiD outside the band and **growing as N falls** | The shape dilution predicts | **Resweep.** This is the case the retrain exists for |
| `delta` changes sign under filtering | The claim itself is filter-dependent | **Resweep**, whatever the magnitude |
| Any of the above but driven by a floor shift | Pool-depth leakage, not dilution | **No resweep.** Diagnose the floor instead |

A resweep is the ~36 h core sweep plus the `C0` ladder, `Cfull-rand`, NB04's 72-run ladder and
NB05 — and, if it is to keep N=16,384, an `ACT_DATASET_PREFIX` raise to ~35,000 that moves the
eval window and invalidates every eval already on disk. The bar is correspondingly high: it is
justified when a **conclusion** changes, not when a number does.

### Caveats to carry into the writeup regardless of outcome

- **That the paper filtered by position type is an inference**, from Table 7 reporting exactly
  five content categories and none of ours mapping to a template token. G.2 describes the filter
  as balancing has-changed across (token, layer) cells; template positions may have fallen out
  of that balancing rather than being excluded by design. Write it as "our pool includes patch
  positions absent from the paper's Table 7 taxonomy".
- **Filtered N and unfiltered N are not the same data**, only the same count. This is a mixture
  comparison, not a row-level ablation.
- **One seed per point on the filtered side.** Sized to decide whether to spend 36 h, not to be
  a result in its own right.
